# GOLD ATP PLAYER-MATCH STATS

## Imports

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [12]:
try:
    spark = SparkSession.builder.appName("fact_player_match_stats").getOrCreate()
except Exception as e:
    print(e)

In [13]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [14]:
# # silver
# tb_player_match = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "silver.tb_atp_tournaments")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# # gold
# tb_tournaments = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_tournaments")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# tb_entry = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_entry")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

# tb_players = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "gold.dim_players")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )


tb_player_match = spark.read.csv(r"../../../data/silver/tb_atp_player_match.csv", sep=',', header=True)
tb_tournaments = spark.read.csv(r"../../../data/gold/dimension/dim_tournaments.csv", sep=',', header=True)
tb_entry = spark.read.csv(r"../../../data/gold/dimension/dim_entry.csv", sep=',', header=True)
tb_players = spark.read.csv(r"../../../data/gold/dimension/dim_players.csv", sep=',', header=True)

## Matches

In [10]:
tb_player_match.limit(1)

MATCH_ID,TOURNEY_ID,PLAYER_ID,PLAYER_IS_WINNER,MATCH_DRAW_SIZE,MATCH_DATE,MATCH_NUM,MATCH_SCORE,MATCH_BEST_OF,MATCH_ROUND,MATCH_DURATION_M,PLAYER_RANK,PLAYER_RANK_PTS,PLAYER_SEED,PLAYER_ENTRY,PLAYER_ACES,PLAYER_DB_FAULTS,PLAYER_SERVE_PTS,PLAYER_1ST_SERVES_IN,PLAYER_1ST_SERVE_PTS_WON,PLAYER_2ND_SERVE_PTS_WON,PLAYER_SERVE_GAMES,PLAYER_BP_SAVED,PLAYER_BP_FACED
1968-2030-238,1968-2030,100172,True,64,19680722,238,6-3 7-5,3,R64,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL


In [ ]:
df = (
    tb_player_match.alias("p_m")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(tb_players.alias("p"), "PLAYER_ID", 'left')
    
    .select(
        f.col("p_m.MATCH_ID"),
        f.col("p.SK_PLAYER"),
        f.col("t.SK_TOURNEY"),

        f.col("p_m.PLAYER_IS_WINNER").cast("int").alias("PLAYER_IS_WINNER"),
        f.coalesce(f.col("p_m.MATCH_DATE").cast("int"), f.lit(-1)).alias("MATCH_DATE"),
        f.coalesce(f.col("p_m.MATCH_NUM").cast("int"), f.lit(-1)).alias("MATCH_NUM"),
        f.coalesce(f.col("p_m.MATCH_SCORE").cast("string"), f.lit('Não Informado')).alias("MATCH_SCORE"), # Provavelmente vou ter que fazer uma dimensao para essa porra aqui
        f.coalesce(f.col("p_m.MATCH_BEST_OF").cast("int"), f.lit(-1)).alias("MATCH_BEST_OF"),
        f.coalesce(f.col("p_m.MATCH_ROUND").cast("string"), f.lit('Não Informado')).alias("MATCH_ROUND"), # Provavelmente vou ter que fazer uma dimensao para essa porra aqui 
        f.coalesce(f.col("p_m.MATCH_DURATION_M").cast("int"), f.lit(-1)).alias("MATCH_DURATION_M"),
        f.coalesce(f.col("p_m.PLAYER_ACES").cast("int"), f.lit(-1)).alias("PLAYER_ACES"), 
        f.coalesce(f.col("p_m.PLAYER_DB_FAULTS").cast("int"), f.lit(-1)).alias("PLAYER_DB_FAULTS"), 
        f.coalesce(f.col("p_m.PLAYER_SERVE_PTS").cast("int"), f.lit(-1)).alias("PLAYER_SERVE_PTS"), 
        f.coalesce(f.col("p_m.PLAYER_1ST_SERVES_IN").cast("int"), f.lit(-1)).alias("PLAYER_1ST_SERVES_IN"), 
        f.coalesce(f.col("p_m.PLAYER_1ST_SERVE_PTS_WON").cast("int"), f.lit(-1)).alias("PLAYER_1ST_SERVE_PTS_WON"), 
        f.coalesce(f.col("p_m.PLAYER_2ND_SERVE_PTS_WON").cast("int"), f.lit(-1)).alias("PLAYER_2ND_SERVE_PTS_WON"), 
        f.coalesce(f.col("p_m.PLAYER_SERVE_GAMES").cast("int"), f.lit(-1)).alias("PLAYER_SERVE_GAMES"), 
        f.coalesce(f.col("p_m.PLAYER_BP_SAVED").cast("int"), f.lit(-1)).alias("PLAYER_BP_SAVED"), 
        f.coalesce(f.col("p_m.PLAYER_BP_FACED").cast("int"), f.lit(-1)).alias("PLAYER_BP_FACED") 
    )
    .distinct()
)

## Validade final dataframe

In [ ]:
if tourney_stats.count() == df.count():
    print('ok')
else: 
    raise

ok


## Save dataframe

### Local

In [ ]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_tournament_stats.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)